Results: CSV
============

This example is a results workflow example, which means it provides tool to set up an effective workflow inspecting
and interpreting the large libraries of modeling results.

In this tutorial, we use the aggregator to load the results of model-fits and output them in a single .csv file.

This enables the results of many model-fits to be concisely summarised and inspected in a single table, which
can also be easily passed on to other collaborators.

__Contents__

- **Model Fit:** Run the shared quick-fit helper if results have not already been created.
- **Workflow Paths:** Set the output path for workflow files.
- **Aggregator:** Load results using the Aggregator.
- **Adding CSV Columns:** Add columns to the CSV file for specific model parameters.
- **Saving the CSV:** Output the CSV file to disk.
- **Customizing CSV Headers:** Customize column headers for readability.
- **Maximum Likelihood Values:** Output maximum likelihood parameter values to the CSV.
- **Errors:** Output parameter error estimates at a given sigma confidence.
- **Column Label List:** Add a label column (e.g. dataset name) to the CSV.
- **Latent Variables:** Add latent (derived) variables to the CSV.
- **Computed Columns:** Add custom computed columns derived from the samples.

__CSV, Png and Fits__

Workflow functionality closely mirrors the `png_make.py` and `fits_make.py`  examples, which load results of
model-fits and output th em as .png files and .fits files to quickly summarise results.

The shared `_quick_fit.py` helper creates these results in `results_folder`. If you have older outputs under `results_folder_csv_png_fits`, use `results_folder` for these examples instead.

__Interferometer__

This script can easily be adapted to analyse the results of charge injection imaging model-fits.

The only entries that needs changing are:

 - `ImagingAgg` -> `InterferometerAgg`.
 - `FitImagingAgg` -> `FitInterferometerAgg`.
 - `Imaging` -> `Interferometer`.
 - `FitImaging` -> `FitInterferometer`.

Quantities specific to an interfometer, for example its uv-wavelengths real space mask, are accessed using the same API
(e.g. `values("dataset.uv_wavelengths")` and `.values{"dataset.real_space_mask")).

__Database File__

The aggregator can also load results from a `.sqlite` database file.

This is beneficial when loading results for large numbers of model-fits (e.g. more than hundreds)
because it is optimized for fast querying of results.

See the package `results/database` for a full description of how to set up the database and the benefits it provides,
especially if loading results from hard-disk is slow.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
except ImportError:
    pass

from autonerves import setup_colab

setup_colab.setup("autogalaxy")

In [ ]:

# from autogalaxy import setup_notebook; setup_notebook()

from pathlib import Path

import autofit as af
import autogalaxy as ag

__Model Fit__

These workflow examples reuse the shared ``_quick_fit.py`` helper instead of
performing model-fits in every script. The helper creates two capped Nautilus
fits, including the latent quantities used below, in ``output/results_folder/``.

Older versions of these workflow examples used ``output/results_folder_csv_png_fits/``;
use ``output/results_folder/`` for the centralized setup here.

In [ ]:
import subprocess
import sys

results_path = Path("output") / "results_folder"
if (
    len(list(results_path.glob("**/image/dataset.fits"))) < 2
    or len(list(results_path.glob("**/files/latent/latent_summary.json"))) < 2
    or len(list(results_path.glob("**/image/fit.png"))) < 2
    or len(list(results_path.glob("**/image/fit.fits"))) < 2
):
    subprocess.run(
        [sys.executable, "scripts/guides/results/_quick_fit.py"],
        check=True,
    )

__Workflow Paths__

The workflow examples are designed to take large libraries of results and distill them down to the key information
required for your science, which are therefore placed in a single path for easy access.

The `workflow_path` specifies where these files are output, in this case the .csv files which summarise the results.

In [ ]:
workflow_path = Path("output") / "results_folder" / "workflow_make_example"

__Aggregator__

Set up the aggregator as shown in `start_here.py`.

In [ ]:
from autofit.aggregator.aggregator import Aggregator

agg = Aggregator.from_directory(
    directory=Path("output") / "results_folder",
)

Extract the `AggregateCSV` object, which has specific functions for outputting results in a CSV format.

In [ ]:
agg_csv = af.AggregateCSV(aggregator=agg)

__Adding CSV Columns__

We first make a simple .csv which contains two columns, corresponding to the inferred median PDF values for
the y centre of the bulge of the galaxy and its effective radius.

To do this, we use the `add_variable` method, which adds a column to the .csv file we write at the end. Every time
we call `add_variable` we add a new column to the .csv file.

Note the API for the `centre`, which is a tuple parameter and therefore needs for `centre_0` to be specified.

The `results_folder` contains two model-fits, meaning that each `add_variable` call adds two rows.

This adds the median PDF value of the parameter to the .csv file, we show how to add other values later in this script.

In [ ]:
agg_csv.add_variable(argument="galaxies.galaxy.bulge.centre.centre_0")
agg_csv.add_variable(argument="galaxies.galaxy.bulge.sersic_index")

__Saving the CSV__

We can now output the results of all our model-fits to the .csv file, using the `save` method.

This will output in your current working directory (e.g. the `autogalaxy_workspace/output/results_folder`)
as a .csv file containing the median PDF values of the parameters, have a quick look now to see the format of 
the .csv file.

In [ ]:
agg_csv.save(path=workflow_path / "csv_simple.csv")

__Customizing CSV Headers__

The headers of the .csv file are by default the argument input above based on the model. 

However, we can customize these headers using the `name` input of the `add_variable` method, for example making them
shorter or more readable.

We recreate the `agg_csv` first, so that we begin adding columns to a new .csv file.

In [ ]:
agg_csv = af.AggregateCSV(aggregator=agg)

agg_csv.add_variable(
    argument="galaxies.galaxy.bulge.centre.centre_0",
    name="bulge_centre._0",
)
agg_csv.add_variable(
    argument="galaxies.galaxy.bulge.sersic_index",
    name="bulge_sersic_index",
)

agg_csv.save(path=workflow_path / "csv_simple_custom_headers.csv")

__Maximum Likelihood Values__

We can also output the maximum likelihood values of each parameter to the .csv file, using the `use_max_log_likelihood`
input.

In [ ]:
agg_csv = af.AggregateCSV(aggregator=agg)

agg_csv.add_variable(
    argument="galaxies.galaxy.bulge.effective_radius",
    name="bulge_effective_radius_max_lh",
    value_types=[af.ValueType.MaxLogLikelihood],
)

agg_csv.save(path=workflow_path / "csv_simple_max_likelihood.csv")

__Errors__

We can also output PDF values at a given sigma confidence of each parameter to the .csv file, using 
the `af.ValueType.ValuesAt3Sigma` input and specifying the sigma confidence.

Below, we add the values at 3.0 sigma confidence to the .csv file, in order to compute the errors you would 
subtract the median value from these values. We add this after the median value, so that the overall inferred
uncertainty of the parameter is clear.

The method below adds three columns to the .csv file, corresponding to the values at the median, lower and upper sigma 
values.

In [ ]:
agg_csv = af.AggregateCSV(aggregator=agg)

agg_csv.add_variable(
    argument="galaxies.galaxy.bulge.effective_radius",
    name="bulge_effective_radius",
    value_types=[af.ValueType.Median, af.ValueType.ValuesAt3Sigma],
)

agg_csv.save(path=workflow_path / "csv_simple_errors.csv")

__Column Label List__

We can add a list of values to the .csv file, provided the list is the same length as the number of model-fits
in the aggregator.

A useful example is adding the name of every dataset to the .csv file in a column on the left, indicating 
which dataset each row corresponds to.

To make this list, we use the `Aggregator` to loop over the `search` objects and extract their `unique_tag`'s, which 
which the helper set from the dataset names. This API can also be used to extract the `name` or `path_prefix`
of the search and build an informative list for the names of the subplots.

We then pass the column `name` and this list to the `add_label_column` method, which will add a column to the .csv file.

In [ ]:
agg_csv = af.AggregateCSV(aggregator=agg)

unique_tag_list = [search.unique_tag for search in agg.values("search")]

agg_csv.add_label_column(
    name="lens_name",
    values=unique_tag_list,
)

agg_csv.save(path=workflow_path / "csv_simple_dataset_name.csv")


__Latent Variables__

Latent variables are not free model parameters but can be derived from the model, and they are described fully in
?.

This example was run with a latent variable called `galaxies.galaxy.bulge.sersic_index_x2`, and below we show that 
this latent variable can be added to the .csv file using the same API as above.

In [ ]:
agg_csv = af.AggregateCSV(aggregator=agg)

agg_csv.add_variable(
    argument="galaxies.galaxy.bulge.sersic_index_x2",
)

agg_csv.save(path=workflow_path / "csv_example_latent.csv")

__Computed Columns__

We can also add columns to the .csv file that are computed from the non-linear search samples (e.g. the nested sampling
samples), for example a value derived from the median PDF instance values of the model.

To do this, we write a function which is input into the `add_computed_column` method, where this function takes the
median PDF instance as input and returns the computed value.

Below, we add a trivial example of a computed column, where the median PDF value that is twice the sersic index of the 
bulge is computed and added to the .csv file.

In [ ]:
agg_csv = af.AggregateCSV(aggregator=agg)


def sersic_index_x2_from(result):

    samples = result.samples

    instance = samples.median_pdf()

    return 2.0 * instance.galaxies.galaxy.bulge.sersic_index


agg_csv.add_computed_column(
    name="bulge_sersic_index_x2_computed",
    compute=sersic_index_x2_from,
)

agg_csv.save(path=workflow_path / "csv_computed_columns.csv")
